# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AxelYoel/FlyRank-AI-Internship---Axel-Yoel-Chandra/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row = one content item (page), summarized over a 90-day window ending 2026-06-30, applied the same for every client — client history length determines whether that window is full, partial, or empty. Grain (one row per page, no duplicates) is verified via the dim_content grain check.

I chose a 90-day window because 57 clients have 90+ days of GSC history, giving a full window. I also included 10 clients with under 90 days of history, flagged via window_days_available so a reviewer can see how many days of data actually back a given page — if a flagged page looks like it needs review, the reviewer can check its window length and judge whether that's a real signal or just noise from a short history.

I excluded 37 clients entirely because they have no GSC history at all — there's no page-level data to review. Including the 10 partial-history clients rather than excluding them too means losing only ~36% of clients (37/104) from analysis, rather than 45% (37+10) — a cost I judged worth accepting rather than losing more coverage, though I haven't measured the business impact directly.

I chose page-level summary over daily rows because daily position is too noisy on low-traffic days — a small daily sample of searches doesn't reliably represent a page's true position, the same small-sample problem I found in Week 1's CTR analysis. This is verified by comparing position stddev on low- vs. high-impression days: 16.6 vs. 6.3 — noise drops substantially with more searches per day, though it doesn't disappear entirely, confirming daily rows carry real noise even if not pure noise.

In [1]:
%pip -q install duckdb
import duckdb
con = duckdb.connect()

In [6]:
import duckdb, os
from google.colab import userdata

hf_token = userdata.get("HF_token")  # exact name you set in Colab Secrets
os.environ["HF_token"] = hf_token

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

In [7]:
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"""
    SELECT COUNT(*) AS n_clients,
           MIN(gsc_data_start) AS earliest_client_start,
           MAX(gsc_data_start) AS latest_client_start
    FROM read_parquet('{rel}/dim_clients.parquet')
""").df()

,n_clients,earliest_client_start,latest_client_start
0,104,2025-01-27,2026-06-02


In [9]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/dim_clients.parquet')").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,is_active,BOOLEAN,YES,None,None,None
2,has_gsc_access,BOOLEAN,YES,None,None,None
3,has_ga4_access,BOOLEAN,YES,None,None,None
4,access_profile,VARCHAR,YES,None,None,None
5,client_created_date,DATE,YES,None,None,None
6,client_updated_date,DATE,YES,None,None,None
7,gsc_data_start,DATE,YES,None,None,None
8,ga4_data_start,DATE,YES,None,None,None


In [10]:
con.sql(f"""
    SELECT client_hash_id, gsc_data_start,
           DATE_DIFF('day', gsc_data_start, DATE '2026-06-30') AS days_of_history
    FROM read_parquet('{rel}/dim_clients.parquet')
    ORDER BY days_of_history
""").df()

,client_hash_id,gsc_data_start,days_of_history
0,client_aef6ffea193da149,2026-06-02,28
1,client_a22068e339bf95f5,2026-05-24,37
2,client_7de9989c909e91a5,2026-05-18,43
3,client_c7c2962f1c9c3089,2026-05-14,47
4,client_1a8bf67cad4ee525,2026-05-11,50
...,...,...,...
99,client_e921cfa93fbe699b,NaT,<NA>
100,client_f0ff30b229fe9b01,NaT,<NA>
101,client_f4fa4e08a1d500d2,NaT,<NA>
102,client_f63f09ff4e81aa58,NaT,<NA>


In [12]:
con.sql(f"""
    SELECT COUNT(*) AS total_clients,
           SUM(CASE WHEN gsc_data_start IS NULL THEN 1 ELSE 0 END) AS n_null_start
    FROM read_parquet('{rel}/dim_clients.parquet')
""").df()

,total_clients,n_null_start
0,104,37.0


In [13]:
con.sql(f"""
    SELECT
      CASE
        WHEN gsc_data_start IS NULL THEN 'no_gsc_history'
        WHEN days_of_history < 90 THEN '<90 days'
        WHEN days_of_history < 180 THEN '90-180 days'
        WHEN days_of_history < 365 THEN '180-365 days'
        ELSE '365+ days'
      END AS history_bucket,
      COUNT(*) AS n_clients
    FROM (
      SELECT client_hash_id, gsc_data_start,
             DATE_DIFF('day', gsc_data_start, DATE '2026-06-30') AS days_of_history
      FROM read_parquet('{rel}/dim_clients.parquet')
    )
    GROUP BY 1
    ORDER BY 1
""").df()

,history_bucket,n_clients
0,180-365 days,31
1,365+ days,9
2,90-180 days,17
3,<90 days,10
4,no_gsc_history,37


In [14]:
con.sql(f"""
    SELECT has_gsc_access,
           SUM(CASE WHEN gsc_data_start IS NULL THEN 1 ELSE 0 END) AS n_null_start,
           COUNT(*) AS n_clients
    FROM read_parquet('{rel}/dim_clients.parquet')
    GROUP BY 1
""").df()

,has_gsc_access,n_null_start,n_clients
0,False,24.0,27
1,<NA>,6.0,10
2,True,7.0,67


In [15]:
con.sql(f"""
    SELECT client_hash_id, has_gsc_access, gsc_data_start, ga4_data_start, is_active
    FROM read_parquet('{rel}/dim_clients.parquet')
    WHERE has_gsc_access = TRUE AND gsc_data_start IS NULL
""").df()

,client_hash_id,has_gsc_access,gsc_data_start,ga4_data_start,is_active
0,client_04660893ae39614a,True,NaT,2026-05-22,True
1,client_19b89ee4fe3db6da,True,NaT,2026-01-09,False
2,client_5781daf723188fe7,True,NaT,NaT,True
3,client_80ee5b7bd5f4eb89,True,NaT,NaT,True
4,client_91c8eb1698a7b352,True,NaT,NaT,True
5,client_e6d6dce7c2fff733,True,NaT,NaT,True
6,client_f0ff30b229fe9b01,True,NaT,NaT,True


In [16]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/dim_content.parquet')").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


In [17]:
con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT content_hash_id) AS unique_content_ids
    FROM read_parquet('{rel}/dim_content.parquet')
""").df()

,total_rows,unique_content_ids
0,519606,519606


In [18]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [22]:
con.sql(f"""
    SELECT content_hash_id, report_date, gsc_avg_position, gsc_clicks, gsc_impressions
    FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
    WHERE content_hash_id IN (
        SELECT content_hash_id
        FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
        WHERE gsc_data_available = TRUE AND gsc_impressions > 0
        GROUP BY 1 HAVING COUNT(*) > 20
        LIMIT 3
    )
    ORDER BY content_hash_id, report_date
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,report_date,gsc_avg_position,gsc_clicks,gsc_impressions
0,content_48a102a5060ed0e2,2026-06-01,52.000000,0,3
1,content_48a102a5060ed0e2,2026-06-02,29.500000,0,4
2,content_48a102a5060ed0e2,2026-06-03,28.333333,0,3
3,content_48a102a5060ed0e2,2026-06-04,32.400000,0,5
4,content_48a102a5060ed0e2,2026-06-05,NaN,0,0
...,...,...,...,...,...
85,content_bf89a688c0ec7cd7,2026-06-26,89.000000,0,1
86,content_bf89a688c0ec7cd7,2026-06-27,66.000000,0,3
87,content_bf89a688c0ec7cd7,2026-06-28,81.666667,0,3
88,content_bf89a688c0ec7cd7,2026-06-29,85.333333,0,3


In [21]:
con.sql(f"""
    SELECT AVG(pos_std) AS avg_within_page_position_stddev
    FROM (
      SELECT content_hash_id, STDDEV(gsc_avg_position) AS pos_std
      FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
      WHERE gsc_data_available = TRUE AND gsc_avg_position > 0
      GROUP BY 1
      HAVING COUNT(*) > 10
    )
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,avg_within_page_position_stddev
0,11.378559


In [23]:
con.sql(f"""
    SELECT
      CASE WHEN gsc_impressions < 10 THEN 'low_impr_days' ELSE 'higher_impr_days' END AS impr_bucket,
      AVG(pos_std) AS avg_position_stddev
    FROM (
      SELECT content_hash_id,
             AVG(gsc_impressions) AS gsc_impressions,
             STDDEV(gsc_avg_position) AS pos_std
      FROM read_parquet('{rel}/fact_content_daily_performance_sample.parquet')
      WHERE gsc_data_available = TRUE AND gsc_avg_position > 0
      GROUP BY 1
      HAVING COUNT(*) > 10
    )
    GROUP BY 1
""").df()

,impr_bucket,avg_position_stddev
0,low_impr_days,16.571946
1,higher_impr_days,6.320960


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.